# 模型規模與計算配額

[完整講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e1-compute-scaling.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/e1_compute_scaling.ipynb)

本練習使用自編小型資料，在 CPU 執行。比較 log-log 斜率與固定配額下的損失最低點。

Hoffmann 等人在 Chinchilla 論文問的是：**固定預訓練算力時，模型參數 $N$ 與訓練 token 數 $D$ 各分配多少，能把語言模型損失降到最低？** 先讀原文第 3 節與 Table 2，再用本課的小模型和圖表重算比例。

## 原論文如何找到算力最優點？

論文用 $C\approx6ND$ 估算密集 Transformer 的訓練運算量。$C$ 是總訓練 FLOPs；固定 $C$ 時，放大 $N$ 就得縮減 $D$。作者用三種方法找最佳配置，各自估計都寫成 $N_{\mathrm{opt}}\propto C^a$、$D_{\mathrm{opt}}\propto C^b$。

| 原文實驗設計 | 找最優解的方式 | Table 2 的 $a$ | Table 2 的 $b$ |
|---|---|---:|---:|
| §3.1 比較訓練曲線的下包絡 | 在同一 FLOPs 預算取達到最低損失的模型與 token 量 | 0.50 | 0.50 |
| §3.2 IsoFLOP（等算力） | 對每個固定預算訓練不同大小的模型，擬合損失曲線的最低點 | 0.49 | 0.51 |
| §3.3 擬合損失函數 | 用已訓練的模型結果擬合 $L(N,D)$，再在預算限制下最小化 | 0.46 | 0.54 |

讀 Figure 3 左圖時，先在每一條**固定 FLOPs**曲線找谷底。中圖和右圖再問，換到更高的 FLOPs 後，谷底對應的參數量和 token 數各移動多少。Table 2 的指數是用這類最優點估出來的。

### 從損失項推算參數與資料的比例

論文 §3.3 使用 $L(N,D)=E+A/N^\alpha+B/D^\beta$。$E$ 是擬合出的損失底限，後兩項分別隨參數或資料增加而減小。把 $D=C/(6N)$ 代回去並對 $N$ 微分，在最優點兩項的邊際變化互相平衡：

$$\frac{\alpha A}{N^\alpha}=\frac{\beta B}{D^\beta}.$$

整理得 $N_{\mathrm{opt}}\propto C^{\beta/(\alpha+\beta)}$、$D_{\mathrm{opt}}\propto C^{\alpha/(\alpha+\beta)}$。附錄用四捨五入的 $\alpha=0.34,\ \beta=0.28$ 寫出損失式，代入可近似得到 0.452 與 0.548。Table 2 的 0.46／0.54 是作者報告的擬合結果。**損失隨 $N,D$ 下降的指數，與最優配置隨 $C$ 擴張的指數，各自回答不同問題。**

### 原文結果怎麼驗算？

第 4 節以與 Gopher 約相當的訓練算力訓練 Chinchilla，使用 70B 參數與 1.4T 訓練 token。Table 6 的 MMLU 5-shot 平均準確率為 Gopher **60.0%**、Chinchilla **67.6%**，差 **7.6 個百分點**。這是論文對該預算和模型設定的實驗證據。訓練配置的結論要和推論延遲、資料品質及規模外推分開解讀。[Chinchilla 原論文](https://arxiv.org/abs/2203.15556)

## 合成暖身與手算

## 用對數座標讀冪律

假設可降低的損失為 $G(N)=3N^{-0.25}$，$N$ 表示模型大小。兩邊取自然對數，就得到 $\ln G=\ln3-0.25\ln N$。橫軸量的是模型大小的倍率，斜率 −0.25 表示模型大小每翻倍，這部分的損失會乘上 $2^{-0.25}\approx0.841$。這個數字描述合成例的冪律。若原始損失含常數底限，要先說清楚畫的是總損失還是扣除底限後的差。

## 固定預算，比較模型和資料的分配

用 $N$ 代表相對模型大小、$D$ 代表相對 token 數。以 $ND=10^6$ 作為簡化配額，並假設教學損失為：

$$L(N,D)=0.3+3N^{-0.25}+1.7D^{-0.25}.$$

| $N$ | $D$ | 配額 $ND$ | 合成損失（約） |
|---:|---:|---:|---:|
| 100 | 10,000 | $10^6$ | 1.419 |
| 1,000 | 1,000 | $10^6$ | 1.136 |
| 10,000 | 100 | $10^6$ | 1.138 |

由第一列到第二列，模型大小增加，模型項的損失下降，但 token 項上升。**應以兩項合計比較同一配額下的結果。** 程式再掃更多候選，找出這個合成公式的最低點。閱讀 Chinchilla 時，查作者如何從真實的模型訓練擬合趨勢。這個小算例的係數、損失與預算單位皆為自編，模型外推與實際硬體成本需要另外驗證。

## 執行環境

Colab 通常已有 NumPy 和 Matplotlib。若要使用本專案規定範圍內的套件，可執行下一格。

In [ ]:
%pip -q install 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 程式驗證

先預測結果，再執行下方的手算核對和視覺化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sizes = np.array([100., 300., 1000., 3000., 10000.])
gap = 3 * sizes ** (-0.25)
slope, intercept = np.polyfit(np.log(sizes), np.log(gap), 1)
np.testing.assert_allclose(slope, -0.25)

budget = 1_000_000
def lesson_loss(model_size, tokens):
    return 0.3 + 3 * model_size ** (-0.25) + 1.7 * tokens ** (-0.25)

token_at_budget = budget / sizes
candidate = np.geomspace(100, 10_000, 151)
losses = lesson_loss(candidate, budget / candidate)
best = int(np.argmin(losses))
np.testing.assert_allclose(lesson_loss(sizes[[0, 2, 4]], token_at_budget[[0, 2, 4]]),
                           [1.418683298, 1.135791722, 1.137587202], atol=1e-6)
print(f"Slope: {slope:.3f}")
print(f"Best grid point: model={candidate[best]:.0f}, tokens={budget/candidate[best]:.0f}, loss={losses[best]:.3f}")
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 3.5))
left.loglog(sizes, gap, "o-")
left.set(xlabel="Model size", ylabel="Synthetic reducible loss", title="Log-log slope = -0.25")
right.semilogx(candidate, losses)
right.scatter(candidate[best], losses[best], color="tab:orange")
right.set(xlabel="Model size at fixed budget", ylabel="Synthetic total loss", title="Compute allocation")
fig.tight_layout()
plt.show()

## 原文對照｜核對 Chinchilla Table 2 的配置斜率

依上方的原文段落先圈出資料集、指標或公式，再用程式核對。圖表依旁邊標註的來源解讀。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Hoffmann et al., Table 2: reported compute-allocation exponents.
approaches = {
    "Training-curve envelope": (0.50, 0.50),
    "IsoFLOP profiles": (0.49, 0.51),
    "Parametric loss fit": (0.46, 0.54),
}
compute_multiplier = np.array([1., 2., 4., 16.])
for label, (a, b) in approaches.items():
    parameter_factor, token_factor = 4 ** a, 4 ** b
    np.testing.assert_allclose(parameter_factor * token_factor, 4)
    print(f"{label}: at 4x FLOPs, N={parameter_factor:.3f}x, D={token_factor:.3f}x")

# Appendix D uses rounded loss-fit exponents; Table 6 reports observed MMLU.
alpha, beta = 0.34, 0.28
print("Allocation powers from rounded appendix coefficients:", beta/(alpha+beta), alpha/(alpha+beta))
np.testing.assert_allclose(67.6 - 60.0, 7.6)
fig, ax = plt.subplots(figsize=(7, 4))
for label, (a, b) in approaches.items():
    ax.loglog(compute_multiplier, compute_multiplier ** a, "o-", label=f"N: {label}")
    ax.loglog(compute_multiplier, compute_multiplier ** b, "--", label=f"D: {label}")
ax.set(xlabel="Compute multiplier", ylabel="Optimal allocation multiplier",
       title="Chinchilla Table 2: ratios implied by the reported exponents")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 自我檢核與練習

**對照原文。** 用 Table 2 的 IsoFLOP 指數算：算力增加四倍時，最佳參數量和 token 數各約變成多少倍？參考答案為 $4^{0.49}\approx1.97$ 與 $4^{0.51}\approx2.03$。再指出 Figure 3 左圖的谷底與右側擬合之間的關係。

把 token 項係數從 1.7 改成 3。先預測較好的分配會往哪裡移，再核對圖表。兩項對稱的情況下，最低點會靠近 $N=D=1000$。

## 來源

- [Chinchilla 原論文](https://arxiv.org/abs/2203.15556)